In [1]:
import alpaca
from alpaca.data.historical import StockHistoricalDataClient
from dotenv import load_dotenv
import os
import pandas as pd
import asyncio

from alpaca.trading import TradingClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime

load_dotenv()

client = StockHistoricalDataClient(api_key=os.environ.get("ALPACA_API_KEY"), secret_key=os.environ.get("ALPACA_SECRET_KEY"))
t_client = TradingClient(api_key=os.environ.get("ALPACA_API_KEY"), secret_key=os.environ.get("ALPACA_SECRET_KEY"))

In [2]:
all_assets = t_client.get_all_assets()

to_get = []
for a in all_assets:
    a = dict(a)
    if a["tradable"]:
        to_get.append(a["symbol"])
len(to_get)

12377

In [3]:
def fetch_list_of_symbols(syms: list, counter) -> None:
    # Creating request object
    request_params = StockBarsRequest(
    symbol_or_symbols=syms,
    timeframe=TimeFrame.Hour,
    start=datetime(2010, 1, 1),
    end=datetime(2025, 1, 1)
    )

    bars = client.get_stock_bars(request_params)

    data = bars.df.reset_index()

    data.to_parquet("data/small_group", index=False, partition_cols=["symbol"])

    print(f"finished i={counter}")

# fetch_list_of_symbols(["AAPL", "AMZN"], -1)

In [4]:
import time

CHUNK_SIZE = 100

In [5]:
to_get = [
    "SPX", "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "TSLA", "AVGO", 
    "CRM", "ADBE", "AMD", "NFLX", "CSCO", "ORCL", "INTC", "QCOM", "IBM", 
    "TXN", "INTU", "UBER", "JPM", "V", "MA", "BAC", "WFC", "GS", "MS", 
    "BRK.B", "BLK", "AXP", "LLY", "UNH", "JNJ", "ABBV", "MRK", "PFE", 
    "TMO", "ABT", "WMT", "COST", "PG", "KO", "PEP", "HD", "MCD", "NKE", 
    "DIS", "XOM", "CVX", "CAT"
]
to_get

['SPX',
 'AAPL',
 'MSFT',
 'NVDA',
 'GOOGL',
 'AMZN',
 'META',
 'TSLA',
 'AVGO',
 'CRM',
 'ADBE',
 'AMD',
 'NFLX',
 'CSCO',
 'ORCL',
 'INTC',
 'QCOM',
 'IBM',
 'TXN',
 'INTU',
 'UBER',
 'JPM',
 'V',
 'MA',
 'BAC',
 'WFC',
 'GS',
 'MS',
 'BRK.B',
 'BLK',
 'AXP',
 'LLY',
 'UNH',
 'JNJ',
 'ABBV',
 'MRK',
 'PFE',
 'TMO',
 'ABT',
 'WMT',
 'COST',
 'PG',
 'KO',
 'PEP',
 'HD',
 'MCD',
 'NKE',
 'DIS',
 'XOM',
 'CVX',
 'CAT']

In [6]:
async def main():
    tasks = []
    for i in range(len(to_get) // CHUNK_SIZE + 1):
        idx = CHUNK_SIZE * i
        chunk = to_get[idx: idx + CHUNK_SIZE]
        print(chunk)

        tasks.append(asyncio.create_task(asyncio.to_thread(fetch_list_of_symbols, chunk, i)))

        # rate limit without blocking event loop
        await asyncio.sleep(32) # asyncio.sleep(CHUNK_SIZE / 200 * 60)

    # Wait for all tasks to finish
    await asyncio.gather(*tasks)

await main()

['SPX', 'AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'TSLA', 'AVGO', 'CRM', 'ADBE', 'AMD', 'NFLX', 'CSCO', 'ORCL', 'INTC', 'QCOM', 'IBM', 'TXN', 'INTU', 'UBER', 'JPM', 'V', 'MA', 'BAC', 'WFC', 'GS', 'MS', 'BRK.B', 'BLK', 'AXP', 'LLY', 'UNH', 'JNJ', 'ABBV', 'MRK', 'PFE', 'TMO', 'ABT', 'WMT', 'COST', 'PG', 'KO', 'PEP', 'HD', 'MCD', 'NKE', 'DIS', 'XOM', 'CVX', 'CAT']
finished i=0


In [ ]:
# tool 25 minutes to get 2016-2025 data of 51 stocks ~ 18,000 hours / entries per... very slow

51